In [ ]:
import pandas as pd
import numpy as np
import os
os.chdir ('C:\\Users\\hamod4691\\Code\\arctic-benthic-biodiversity\\All data')

In [ ]:
# Read the column mapping
mapping = pd.read_csv('OBIS columns.csv', sep=';')

print(f"Total columns to create: {len(mapping)}")
print(f"\nColumn mapping:")
print(mapping.head(10))

In [ ]:
# Read the OBIS parquet file
obis_df = pd.read_parquet('../OBIS/obis_subset_merged_FP.parquet')

print(f"OBIS data shape: {obis_df.shape}")
print(f"Number of records: {len(obis_df):,}")

In [ ]:
# Create the new dataframe with mapped columns
all_merged = pd.DataFrame(index=obis_df.index)

for idx, row in mapping.iterrows():
    new_col = row['New file']
    obis_col = row['OBIS']
    
    # Handle special case: sourceDatabase should be filled with literal "OBIS"
    if new_col == 'sourceDatabase':
        all_merged[new_col] = 'OBIS'
    
    # Handle empty mapping - create empty column
    elif pd.isna(obis_col) or obis_col.strip() == '':
        all_merged[new_col] = None
    
    # Handle multiple source columns (use first available)
    elif ',' in obis_col:
        sources = [c.strip() for c in obis_col.split(',')]
        # Use the first existing source column
        for source in sources:
            if source in obis_df.columns:
                all_merged[new_col] = obis_df[source]
                break
        else:
            # None of the sources exist, create empty column
            all_merged[new_col] = None
    
    # Handle single source column
    else:
        obis_col = obis_col.strip()
        if obis_col in obis_df.columns:
            all_merged[new_col] = obis_df[obis_col]
        else:
            # Source column doesn't exist, create empty column
            all_merged[new_col] = None

print(f"Created dataframe with {len(all_merged.columns)} columns and {len(all_merged):,} rows")
print(f"\nColumn names:")
print(all_merged.columns.tolist())

In [ ]:
# Verify the data
print("Sample of the merged data:")
print(all_merged.head())

print(f"\nData types:")
print(all_merged.dtypes)

print(f"\nNull counts:")
null_counts = all_merged.isnull().sum()
print(null_counts[null_counts > 0].sort_values(ascending=False))

In [ ]:
# Save to parquet file
output_path = 'all_merged.parquet'
all_merged.to_parquet(output_path, index=False)

print(f"Successfully saved to {output_path}")
print(f"File contains {len(all_merged):,} records with {len(all_merged.columns)} columns")